# Stage 02 · Step 0 — Generate multi-τ training corpus

The supervised RUL head benefits from telemetry observed under several maintenance schedules, not just the single baseline τ already in `data/fleet_baseline.parquet`. This notebook draws K τ vectors via Latin-Hypercube sampling over the same ranges Stage 01 explores and runs the SDG simulator on the **train printers (id 0..69)** for each.

Outputs:
- `data/policy_runs/policy_{k:03d}.parquet` — one parquet per τ schedule (train printers only, RUL labels included).
- `data/policy_runs/manifest.json` — index of τ values per file.

Skip this notebook if the corpus already exists; the SSL pretraining notebook will fall back to `fleet_baseline.parquet` if no policy runs are found.

In [1]:
from __future__ import annotations
import json
from pathlib import Path

import numpy as np
import pyarrow.parquet as pq
from scipy.stats import qmc

from ml_models.lib.data import TRAIN_PRINTERS
from ml_models.lib.env_runner import default_dates, run_with_tau
from ml_models import PROJECT_ROOT
from sdg.generate import load_configs
from sdg.labels import add_rul_labels
from sdg.schema import COMPONENT_IDS, table_from_dataframe

POLICY_DIR = PROJECT_ROOT / 'data/policy_runs'
POLICY_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = POLICY_DIR / 'manifest.json'

In [2]:
TAU_RANGES = {
    'C1': (50.0, 2_000.0),
    'C2': (500.0, 20_000.0),
    'C3': (24.0, 500.0),
    'C4': (100.0, 2_000.0),
    'C5': (500.0, 8_000.0),
    'C6': (1_000.0, 20_000.0),
}
K = 5  # number of policy runs; set higher (e.g. 30) for stronger surrogate generalisation
SEED = 17

sampler = qmc.LatinHypercube(d=len(TAU_RANGES), seed=SEED)
unit = sampler.random(K)
tau_schedules: list[dict[str, float]] = []
for row in unit:
    schedule = {}
    for i, (component_id, (low, high)) in enumerate(TAU_RANGES.items()):
        schedule[component_id] = float(np.exp(np.log(low) + row[i] * (np.log(high) - np.log(low))))
    tau_schedules.append(schedule)
tau_schedules

[{'C1': 512.6794318428621,
  'C2': 8492.592534298488,
  'C3': 31.394740120707354,
  'C4': 146.02573139348334,
  'C5': 7101.119224365106,
  'C6': 8718.269048285229},
 {'C1': 1458.2801119642934,
  'C2': 666.0400359507457,
  'C3': 174.1787573923656,
  'C4': 328.4229803601047,
  'C5': 1316.554522486139,
  'C6': 2307.860863841853},
 {'C1': 205.57190441316095,
  'C2': 2190.3350070492033,
  'C3': 89.5228191916952,
  'C4': 1074.6184865850778,
  'C5': 1926.481256456714,
  'C6': 1263.7234369298835},
 {'C1': 454.97406372544725,
  'C2': 17524.650835816577,
  'C3': 452.3467574192089,
  'C4': 457.5238920433893,
  'C5': 3355.1961405229345,
  'C6': 4602.86784969408},
 {'C1': 53.048711401544956,
  'C2': 1198.6227249135914,
  'C3': 63.37332544337184,
  'C4': 1770.7862451810777,
  'C5': 713.6752118434381,
  'C6': 11932.276480808643}]

In [3]:
components_cfg, couplings_cfg, cities_cfg = load_configs()
DATES = default_dates()
manifest = []
for k, schedule in enumerate(tau_schedules):
    output_path = POLICY_DIR / f'policy_{k:03d}.parquet'
    if output_path.exists():
        print(f'[{k}] cached:', output_path)
        manifest.append({'k': k, 'tau': schedule, 'path': output_path.as_posix()})
        continue
    print(f'[{k}] simulating', schedule)
    df = run_with_tau(
        schedule,
        printer_ids=TRAIN_PRINTERS,
        dates=DATES,
        components_cfg=components_cfg,
        couplings_cfg=couplings_cfg,
        cities_cfg=cities_cfg,
    )
    table = table_from_dataframe(df, include_rul=False)
    pq.write_table(table, output_path, compression='snappy', version='2.6')
    add_rul_labels(output_path)
    manifest.append({'k': k, 'tau': schedule, 'path': output_path.as_posix()})
    print(f'[{k}] wrote {output_path} ({output_path.stat().st_size / 1e6:.1f} MB)')

[0] simulating {'C1': 512.6794318428621, 'C2': 8492.592534298488, 'C3': 31.394740120707354, 'C4': 146.02573139348334, 'C5': 7101.119224365106, 'C6': 8718.269048285229}
[0] wrote /home/sterry/Desktop/projects/hackupc2026/data/policy_runs/policy_000.parquet (20.9 MB)
[1] simulating {'C1': 1458.2801119642934, 'C2': 666.0400359507457, 'C3': 174.1787573923656, 'C4': 328.4229803601047, 'C5': 1316.554522486139, 'C6': 2307.860863841853}
[1] wrote /home/sterry/Desktop/projects/hackupc2026/data/policy_runs/policy_001.parquet (21.3 MB)
[2] simulating {'C1': 205.57190441316095, 'C2': 2190.3350070492033, 'C3': 89.5228191916952, 'C4': 1074.6184865850778, 'C5': 1926.481256456714, 'C6': 1263.7234369298835}
[2] wrote /home/sterry/Desktop/projects/hackupc2026/data/policy_runs/policy_002.parquet (20.5 MB)
[3] simulating {'C1': 454.97406372544725, 'C2': 17524.650835816577, 'C3': 452.3467574192089, 'C4': 457.5238920433893, 'C5': 3355.1961405229345, 'C6': 4602.86784969408}
[3] wrote /home/sterry/Desktop/pro

In [4]:
with MANIFEST_PATH.open('w', encoding='utf-8') as handle:
    json.dump({'tau_ranges': TAU_RANGES, 'seed': SEED, 'runs': manifest}, handle, indent=2)
print('Wrote manifest:', MANIFEST_PATH)
print(f'{len(manifest)} policy runs covering printer_ids {TRAIN_PRINTERS[0]}..{TRAIN_PRINTERS[-1]}')

Wrote manifest: /home/sterry/Desktop/projects/hackupc2026/data/policy_runs/manifest.json
5 policy runs covering printer_ids 0..69
